# Probeprüfung AP02: Datenbeschaffung, Datenaufbereitung & explorative Datenanalyse (Probeprüfung – Musterlösung)

Aufgebaut wie die Prüfung *AP02* — gleiche Teile, gleiche Punkte, gleiches Bewertungsschema —, aber mit den Daten des
fiktiven Velo-Shops **VeloMarkt** statt PhoneMarkt.

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

| Teil | Thema | Aufgaben | Punkte |
|---|---|---|---|
| A | Konzeptfragen: Datenaufbereitung & explorative Datenanalyse | A1–A3 | 12 |
| B | Fehleranalyse Python-Code (Shop-Daten) | B1–B2 | 16 |
| C | Python-Code schreiben: Web Scraping, Datenaufbereitung & EDA | C1–C3 | 12 |
| **Total** | | **8 Aufgaben** | **40** |

### Bewertungsschema

| Kategorie | Beschreibung | Bewertung | 4 Pkt | 8 Pkt |
|---|---|---|---|---|
| Korrekt | Vollständig und fachlich einwandfrei; Code liefert das richtige Ergebnis | 100% | 4 | 8 |
| Minimale Mängel | Kleine Fehler (z. B. falsche Sortierung, fehlende Rundung, fehlende Achsenbeschriftung) | 75% | 3 | 6 |
| Mittlere Mängel | Grössere Fehler (z. B. fehlende Spalten, falsche Filterung) oder Begründung fehlt | 50% | 2 | 4 |
| Gravierende Mängel | Ansatz erkennbar, aber im Kern falsch (z. B. falscher Join-Typ, falsche Datei) | 25% | 1 | 2 |
| Nicht lauffähig | Antwort fehlt oder Code nicht lauffähig | 0% | 0 | 0 |

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege gelten als korrekt.

*Empfohlene Bearbeitungszeit: 60 Minuten. Open book, aber ohne KI — wie in der Prüfung.*

## Hinweise zu den Daten

Alle Daten stammen vom **fiktiven** Schweizer Online-Shop **«VeloMarkt»** und liegen lokal in `Data/velomarkt/`.
Es ist kein Internetzugriff nötig. Spaltenbeschreibungen: `Data/velomarkt/README.md`.

| Datei | Format | Inhalt |
|---|---|---|
| `html/shop_page_1.html` … `shop_page_3.html` | HTML | gespeicherte Produktlisten-Seiten (Paginierung) |
| `html/product_detail.html` | HTML | Detailseite eines E-Bikes (Preisverlauf, technische Daten, Lieferung) |
| `reviews.json` | JSON | gespeicherte Antwort der Reviews-API: Bewertungen pro `product_id` |
| `velos_clean.csv` | CSV | aufbereitete Shop-Daten — **inhaltlich nicht geprüft** |

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import re
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Daten von «VeloMarkt» liegen hier:
DATA = 'Data/velomarkt'
print(os.getcwd())

### Produktlisten-Seiten scrapen → `df_raw`

Der folgende Code liest die drei gespeicherten Produktlisten-Seiten mit BeautifulSoup ein. Jede Produktkarte
(`<div class="product-card">`) wird zu einer Zeile im DataFrame `df_raw`. Die Werte werden **als Text** übernommen
(Spalten mit Endung `_raw`) und noch nicht bereinigt.

In [ ]:
# Initialize list for the extracted rows
rows = []

# Loop over the three listing pages
for page in [1, 2, 3]:

    # Read html from file
    with open(f'{DATA}/html/shop_page_{page}.html', 'r', encoding='utf-8') as file:
        html_content = file.read()

    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Locate the product cards and extract values
    for card in soup.select('div.product-card'):
        row_data = [card['data-product-id'],
                    card.select('span.brand')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.model')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.category')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.weight')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.battery')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.range')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.price')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.availability')[0].get_text().replace('\xa0', '').strip()]
        rows.append(row_data)

# Create a DataFrame
df_raw = pd.DataFrame(rows, columns=['product_id', 'brand', 'model', 'category', 'weight_raw',
                                     'battery_raw', 'range_raw', 'price_raw', 'availability'])

# Show dimensions and first rows
print('Number of rows and columns:', df_raw.shape)
df_raw.head()

### Aufbereitete Shop-Daten laden → `df_clean`

In [ ]:
# Import data from csv to pandas dataframe named 'df_clean'
df_clean = pd.read_csv(f'{DATA}/velos_clean.csv')

# Show dimensions and first rows
print('Number of rows and columns:', df_clean.shape)
df_clean.head()

> **Hinweis:** Alle Aufgaben sind voneinander unabhängig. Verändern Sie `df_raw` und `df_clean` bitte nicht direkt,
> sondern arbeiten Sie in jeder Aufgabe mit einem eigenen DataFrame (z. B. `df_B1`, `df_C2`).

---
**▶ Ab hier beginnt die bewertete Probeprüfung — 40 Punkte**

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den Shop-Daten von «VeloMarkt» werden nicht gewertet.*

### A1 (4 Punkte)

In `velos_clean.csv` fehlt `weight_kg` bei 4 Velos: Der Lieferant hat für einzelne, zufällig verteilte Modelle kein
Datenblatt geliefert. `battery_wh` fehlt bei 59 der 120 Velos — es sind genau die Velos ohne Elektromotor
(Rennvelo, Gravel, City, Kindervelo).

a) Um welchen Mechanismus fehlender Werte handelt es sich jeweils (MCAR, MAR oder MNAR)? Begründen Sie.

b) Ein Analyst ersetzt alle fehlenden `battery_wh` durch den Mittelwert (707 Wh) und berichtet anschliessend die
durchschnittliche Akkugrösse **aller** Velos. Beurteilen Sie dieses Vorgehen.

**Ihre Antwort:**

a) `weight_kg`: **MCAR** — das Fehlen hängt weder vom Gewicht noch von anderen Merkmalen ab (zufällig fehlende
Datenblätter). `battery_wh`: Das Fehlen hängt vollständig von der **beobachteten** Kategorie ab (alle Velos ohne
Motor), formal also **MAR**. Genauer ist es ein **strukturell** fehlender Wert: Ein Rennvelo hat keinen Akku, der Wert
ist nicht «unbekannt», sondern «nicht anwendbar».

b) Fachlich falsch: Damit würden 59 Velos ohne Motor ein fiktiver 707-Wh-Akku zugeschrieben. Eine «durchschnittliche
Akkugrösse aller Velos» ist sinnlos. Richtig ist, Akkukennzahlen nur für E-Bikes zu berichten (61 Velos) und die
fehlenden Werte nicht zu imputieren, z. B. mit einer Spalte `ebike` (ja/nein) als Filter.

### A2 (4 Punkte)

Für die Spalte `weight_kg` in `velos_clean.csv` liefert pandas folgendes Ergebnis:

```
df_clean['weight_kg'].describe()          df_clean['weight_kg'].skew()
count    116.00                            0.00
mean      17.30
std        7.55
min        6.80
25%        9.70
50%       16.40
75%       24.42
max       28.50
```

Eine Kollegin schreibt in den Bericht: *«Das typische Velo im Sortiment wiegt rund 17 kg; die Gewichte sind
symmetrisch verteilt.»* Beurteilen Sie diese Aussage anhand der Kennzahlen und nennen Sie eine geeignete Grafik, um Ihre
Einschätzung zu überprüfen.

**Ihre Antwort:**

Die Schiefe von 0 zeigt nur Symmetrie, nicht, dass die Werte um den Mittelwert konzentriert sind. Der Interquartilsabstand
ist mit 9.7 bis 24.4 kg sehr breit, und die Standardabweichung (7.55) ist im Verhältnis zum Mittelwert gross. Das
spricht für **zwei Gruppen**: leichte Velos ohne Motor (ca. 7–17 kg) und schwere E-Bikes (ca. 21–28 kg). Die Verteilung
ist **bimodal**, und im Bereich um 17 kg liegen kaum Velos. Ein «typisches Velo mit 17 kg» gibt es also nicht.
Prüfen lässt sich das mit einem **Histogramm** (zwei Gipfel), am besten zusätzlich getrennt nach E-Bike ja/nein.
Berichten sollte man Kennzahlen pro Gruppe.

### A3 (4 Punkte)

Ein Kollege berechnet für alle E-Bikes mit Preisangabe (n = 55) die Korrelation zwischen Akkugrösse und Preis:

```
df[['battery_wh', 'price_chf']].corr()     # Ergebnis: r = -0.10
```

Er folgert: *«Die Akkugrösse hat keinen Einfluss auf den Preis — grössere Akkus muss man im Marketing nicht bewerben.»*

Nennen und begründen Sie **zwei** fachliche Einwände gegen diese Vorgehensweise bzw. Schlussfolgerung.

**Ihre Antwort:**

1. **Ausreisser verzerren Pearson:** `velos_clean.csv` enthält den Tippfehler `VM-30032` (E-City, 30'981 statt 3'098).
   Ein einzelner Extremwert kann r stark verändern. Vorher bereinigen oder robust mit Spearman rechnen.
2. **Nicht zufällig fehlende Preise (MNAR):** Die teuersten E-Bikes (ab CHF 9'000, «Preis auf Anfrage») fehlen in der
   Berechnung. Der Preisbereich ist abgeschnitten, ein Zusammenhang kann dadurch verdeckt werden.
3. **Vermischte Gruppen / Störvariable und Kausalität:** E-City, E-Trekking, E-MTB und S-Pedelec haben sehr
   unterschiedliche Preisniveaus. Ein Zusammenhang **innerhalb** einer Kategorie kann im Gesamtwert verschwinden.
   Pearson misst nur lineare Zusammenhänge, und aus einer Korrelation (oder ihrem Fehlen) lässt sich keine Aussage über
   die Wirkung auf Kaufentscheide ableiten.

*(Zwei gut begründete Einwände genügen.)*

---
## Teil B – Fehleranalyse Python-Code (16 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Kennzahlen für das Sortiment aus den gescrapten Rohdaten `df_raw`: Anzahl Velos, Anzahl Velos ohne
Preisangabe und Median-Preis in CHF.

```python
# Select columns for the analysis
df_B1 = df_raw[['product_id', 'brand', 'model', 'category', 'price_raw']]

# Extract numerical values from price_raw
price = []
for i in df_B1['price_raw']:
    d1 = re.findall(r"\d+", i)
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df_B1['price_chf'] = pd.Series(price, dtype="Int64")

# Show key figures
print('Anzahl Velos:             ', df_B1.shape[0])
print('Anzahl ohne Preisangabe:  ', df_B1['price_chf'].isnull().sum())
print('Median-Preis (CHF):       ', df_B1['price_chf'].median())
```

Tatsächliches Ergebnis:

```
Anzahl Velos:              126
Anzahl ohne Preisangabe:   7
Median-Preis (CHF):        4.0
```

Der Shop gibt auf jeder Seite an, dass er **120 Artikel** führt. Beispiele für Werte in `price_raw`: `CHF899.–`,
`CHF3'898.–`, `Preis auf Anfrage`.

**Aufgabe:** Schreiben Sie den korrigierten Code (die Kommentare des ursprünglichen Codes sind als Gerüst vorgegeben),
sodass die Kennzahlen stimmen. Begründen Sie jede Änderung gegenüber dem ursprünglichen Code.

**Begründung der Änderungen:**

1. **Duplikate:** Die Paginierung überlappt — 6 Velos stehen auf Seite 2 und 3 und werden doppelt gescrapt
   (126 statt 120 Zeilen). Lösung: `drop_duplicates(subset='product_id')`. Dadurch sinkt auch die Zahl der
   Velos ohne Preis von 7 auf 6, weil ein doppelt erfasstes Velo «Preis auf Anfrage» hatte.
2. **Index neu setzen:** Nach `drop_duplicates()` hat der Index Lücken. `pd.Series(price)` hat den Index 0–119 und
   würde beim Zuweisen nach Index (nicht nach Position) ausgerichtet, die Preise landeten in falschen Zeilen.
   Lösung: `reset_index(drop=True)`.
3. **Tausender-Apostroph:** `re.findall(r"\d+", "CHF3'898.–")` liefert `['3', '898']`, mit `[0]` wird nur `3`
   übernommen. Daher der unsinnige Median von 4. Lösung: den Apostroph vor der Suche entfernen
   (`i.replace("'", "")`).
4. *(Stil)* `except:` fängt jeden Fehler ab. Gezielt ist `except IndexError:`, der Fall «keine Zahl gefunden».

**Korrigierter Code:**

In [ ]:
# Select columns for the analysis (Duplikate entfernen, Index neu setzen)
df_B1 = (df_raw[['product_id', 'brand', 'model', 'category', 'price_raw']]
         .drop_duplicates(subset='product_id')
         .reset_index(drop=True))

# Extract numerical values from price_raw (Tausender-Apostroph entfernen)
price = []
for i in df_B1['price_raw']:
    d1 = re.findall(r"\d+", i.replace("'", ""))
    try:
        d2 = int(d1[0])
    except IndexError:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df_B1['price_chf'] = pd.Series(price, dtype="Int64")

# Show key figures
print('Anzahl Velos:             ', df_B1.shape[0])
print('Anzahl ohne Preisangabe:  ', df_B1['price_chf'].isnull().sum())
print('Median-Preis (CHF):       ', df_B1['price_chf'].median())

In [ ]:
# Kontrolle der Musterlösung
assert df_B1.shape[0] == 120 and df_B1['price_chf'].isnull().sum() == 6 and df_B1['price_chf'].median() == 3698.0

### B2 (8 Punkte)

**Ziel:** Herausfinden, welche Marke im Sortiment am teuersten bzw. am günstigsten ist (auf Basis von `df_clean`).

```python
# Select columns for the analysis
df_B2 = df_clean[['brand', 'price_chf']]

# Price per brand
df_B2.groupby(['brand']).mean().round(2).sort_values(by='price_chf', ascending=False)
```

Tatsächliches Ergebnis:

```
                price_chf
brand
Stromer           7849.20
Specialized       5518.00
Scott             4789.67
Flyer             4498.00
Canyon            4356.33
Tour de Suisse    4113.33
Trek              2540.86
Cube              2296.82
```

Der Einkaufsleiter ist skeptisch: *«Tour de Suisse ist unsere Einstiegsmarke mit vielen günstigen City-Velos — teurer
als Trek und Cube kann nicht stimmen.»*

**Aufgabe:** Untersuchen Sie die Daten mit geeigneten EDA-Methoden, finden Sie die Ursache für das irreführende Ergebnis
und schreiben Sie den korrigierten Code. Begründen Sie jede Änderung gegenüber dem ursprünglichen Code.

**Begründung der Änderungen:**

**Untersuchung:** Ein Boxplot der Preise pro Marke bzw. `groupby('brand')['price_chf'].describe()` zeigt bei
«Tour de Suisse» einen extremen Ausreisser: `VM-30032` (Basel EX, E-City) kostet 30'981 CHF. Der Median der Marke liegt
bei nur 1'598 CHF, das Maximum aller anderen E-City-Velos weit darunter. Die IQR-Regel (obere Grenze ca. 9'300 CHF)
markiert genau diesen Wert. Der Vergleich mit dem Shop (`df_raw`: «CHF3'098.–») zeigt einen Tippfehler, eine Ziffer
zu viel.

**Änderungen:**
1. Den nachweislich falschen Wert korrigieren (3'098) oder, falls der richtige Wert unbekannt ist, auf `NaN` setzen.
   Nicht einfach ungeprüft löschen.
2. Zusätzlich den **Median** pro Marke ausgeben: Er ist robust gegen Ausreisser und bei schiefen Preisverteilungen
   aussagekräftiger als der Mittelwert. Mit beiden Kennzahlen fallen solche Fehler sofort auf.

Ergebnis: «Tour de Suisse» ist mit Mittelwert ca. 1'790 CHF (Median 1'598) die günstigste Marke.

**Korrigierter Code:**

In [ ]:
# EDA: Ausreisser pro Marke sichtbar machen
sns.boxplot(data=df_clean, x='price_chf', y='brand')
plt.title('Preise pro Marke')
plt.xlabel('Preis (CHF)')
plt.ylabel('Marke')
plt.show()
display(df_clean.loc[df_clean['price_chf'] > 20000, ['product_id', 'brand', 'model', 'category', 'price_chf']])
display(df_raw.loc[df_raw['product_id'] == 'VM-30032', ['product_id', 'price_raw']])

# Select columns for the analysis (Kopie, Tippfehler korrigieren: Shop zeigt CHF 3'098.–)
df_B2 = df_clean[['product_id', 'brand', 'price_chf']].copy()
df_B2.loc[df_B2['product_id'] == 'VM-30032', 'price_chf'] = 3098.0

# Price per brand (Mittelwert und Median)
df_B2.groupby(['brand'])['price_chf'].agg(['mean', 'median']).round(2).sort_values(by='mean', ascending=False)

In [ ]:
# Kontrolle der Musterlösung
ergebnis_B2 = df_B2.groupby('brand')['price_chf'].mean().sort_values(ascending=False)
assert ergebnis_B2.index[-1] == 'Tour de Suisse' and abs(ergebnis_B2['Tour de Suisse'] - 1789.75) < 0.01

---
## Teil C – Python-Code schreiben (12 Punkte)

Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.

### C1 (4 Punkte)

Die Datei `html/product_detail.html` enthält die Detailseite des Flyer Upstreet 5.23. Das Gerüst unten liest die
HTML-Datei ein und überführt eine Tabelle Zeile für Zeile in den DataFrame `df_C1` mit den Spalten `merkmal` und `wert`.

Ergänzen Sie die beiden mit `# TODO` markierten Stellen:

1. Wählen Sie die Tabelle **«Technische Daten»** aus. Alle drei Tabellen der Seite haben dieselbe Klasse.
2. Entfernen Sie die Zwischenüberschriften der Tabelle (z. B. «Antrieb»). Sie sind **keine** Merkmale und dürfen im
   Ergebnis nicht vorkommen.

Begründen Sie, wie Sie die richtige Tabelle auswählen und die Zwischenüberschriften entfernen.

**Begründung der Vorgehensweise:**

Alle drei Tabellen haben die Klasse `data-table`. Eindeutig ist nur die vorangehende Überschrift. Deshalb suche ich das
`<h2>` mit dem Text «Technische Daten» und nehme mit `.find_next('table')` die darauf folgende Tabelle. Das ist robuster
als ein Positionsindex (`find_all('table')[1]`). Die Zwischenüberschriften bestehen aus nur **einer** Zelle
(`<td colspan="2" class="section">`), darum ist `wert` in diesen Zeilen `None`. Mit `dropna(subset=['wert'])` werden
genau diese Zeilen entfernt, es bleiben 9 Merkmale.

**Python-Code:**

In [ ]:
# Read html from file
with open(f'{DATA}/html/product_detail.html', 'r', encoding='utf-8') as file:
    html_content = file.read()

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Locate the table «Technische Daten» (über die Überschrift, nicht über die Position)
table = soup.find('h2', string='Technische Daten').find_next('table')

# Extract table rows
rows = []
for row in table.find_all('tr'):
    rows.append([cell.get_text().strip() for cell in row.find_all('td')])

# Create a DataFrame
df_C1 = pd.DataFrame(rows, columns=['merkmal', 'wert'])

# Remove the sub-headings (Zeilen mit nur einer Zelle -> wert ist None)
df_C1 = df_C1.dropna(subset=['wert']).reset_index(drop=True)

# Show the result
df_C1

In [ ]:
# Kontrolle der Musterlösung
assert len(df_C1) == 9 and df_C1['merkmal'].tolist()[0] == 'Motor'

### C2 (4 Punkte)

Teilen Sie die Velos in `df_clean` anhand von `price_chf` mit `pd.qcut()` in **vier gleich grosse** Preisklassen mit
den Labels `Einstieg`, `Mittel`, `Gehoben` und `Premium` ein (neue Spalte `segment`).

Erstellen Sie anschliessend mit `pd.crosstab()` eine Kontingenztabelle mit der Anzahl Velos pro Kategorie
(`category`, Zeilen) und Preisklasse (`segment`, Spalten) inklusive Summen (`margins=True`). Begründen Sie, wie
`pd.qcut()` die Klassengrenzen bestimmt und was mit den Velos ohne Preis geschieht.

**Begründung der Vorgehensweise:**

`pd.qcut(..., q=4)` setzt die Klassengrenzen an die **Quartile** der Preise (25 %, 50 %, 75 %), sodass jede Klasse etwa
gleich viele Velos enthält (je ca. 28–29 von 114). Die Klassen sind dadurch unterschiedlich **breit**. Velos ohne Preis
(6 × «Preis auf Anfrage») erhalten `NaN` als Segment und werden von `pd.crosstab()` nicht gezählt. Die Summe ist
deshalb 114, nicht 120. Weil diese Velos die teuersten sind, ist die Klasse «Premium» eigentlich unterbesetzt.

**Python-Code:**

In [ ]:
df_C2 = df_clean.copy()
df_C2['segment'] = pd.qcut(df_C2['price_chf'], q=4, labels=['Einstieg', 'Mittel', 'Gehoben', 'Premium'])

print(df_C2['segment'].value_counts(dropna=False))
pd.crosstab(df_C2['category'], df_C2['segment'], margins=True)

In [ ]:
# Kontrolle der Musterlösung
kreuz_C2 = pd.crosstab(df_C2['category'], df_C2['segment'], margins=True)
assert kreuz_C2.loc['All', 'All'] == 114 and df_C2['segment'].isnull().sum() == 6

### C3 (4 Punkte)

Lesen Sie die Kundenbewertungen aus `reviews.json` in einen DataFrame ein und verbinden Sie ihn mit `.merge()` über
`product_id` mit `df_clean`. Berücksichtigen Sie nur Velos mit **mindestens 5 Bewertungen** (`review_count`).
Berechnen Sie die **Spearman-Rangkorrelation** zwischen `price_chf` und `avg_score` (gerundet auf 2 Dezimalstellen) und
erstellen Sie ein Streudiagramm (Scatter Plot) mit beschrifteten Achsen.

Begründen Sie, welche Velos in die Berechnung eingehen, und interpretieren Sie das Ergebnis in 1–2 Sätzen.

**Begründung der Vorgehensweise:**

Mit `how='inner'` bleiben nur Velos, die **sowohl** im Sortiment als auch in `reviews.json` vorkommen (95). Bewertungen
von 4 nicht mehr geführten Produkten und 25 Velos ohne Bewertung fallen weg. Der Filter `review_count >= 5` entfernt
Durchschnitte aus nur 1–3 Bewertungen, die wenig verlässlich sind (87). `corr()` ignoriert zusätzlich die Velos ohne
Preis, es gehen **82** Velos in die Berechnung ein. Spearman (rangbasiert) ist robust gegen den Preis-Ausreisser und
die schiefe Verteilung. **Interpretation:** ρ ≈ 0.35 ist ein schwacher bis mittlerer positiver, monotoner Zusammenhang.
Teurere Velos werden tendenziell etwas besser bewertet, der Preis erklärt die Zufriedenheit aber nur zu einem kleinen
Teil, und Kausalität ist damit nicht belegt.

**Python-Code:**

In [ ]:
df_reviews = pd.read_json(f'{DATA}/reviews.json')

df_C3 = df_clean.merge(df_reviews, on='product_id', how='inner')
df_C3 = df_C3[df_C3['review_count'] >= 5]
print('Velos nach merge und Filter:', len(df_C3), '| davon mit Preis:', df_C3['price_chf'].notnull().sum())

rho = df_C3[['price_chf', 'avg_score']].corr(method='spearman').iloc[0, 1]
print('Spearman-Rangkorrelation:', round(rho, 2))

sns.scatterplot(data=df_C3, x='price_chf', y='avg_score')
plt.title('Preis und Kundenbewertung (mind. 5 Bewertungen)')
plt.xlabel('Preis (CHF)')
plt.ylabel('Durchschnittliche Bewertung (1–5 Sterne)')
plt.show()

In [ ]:
# Kontrolle der Musterlösung
assert df_C3['price_chf'].notnull().sum() == 82 and round(rho, 2) == 0.35

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')